In [2]:
%%capture
!pip install "protobuf<5" vllm transformers langgraph langsmith langchain langchainhub -U -q

In [3]:
import os
from kaggle_secrets import UserSecretsClient
from IPython.display import display, Markdown
secrets = UserSecretsClient()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = secrets.get_secret("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "ArXiv-Agent-Research"

os.environ["LANGCHAIN_INGEST_MULTIPART"] = "false"

In [4]:
import sys
import psycopg2
repo_path = f"/kaggle/working/Summarization-scientific-articles"

if not os.path.exists(repo_path):
    !git clone https://github.com/fluloeo/Summarization-scientific-articles.git

if repo_path not in sys.path:
    sys.path.append(repo_path)

Cloning into 'Summarization-scientific-articles'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 159 (delta 77), reused 36 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (159/159), 738.75 KiB | 4.48 MiB/s, done.
Resolving deltas: 100% (77/77), done.


In [5]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
model_id = "Qwen/Qwen3-4B-Instruct-2507"
llm = LLM(
    model= model_id,
    tensor_parallel_size=2,  
    max_model_len=16384,     
    gpu_memory_utilization=0.85,
    attention_backend="TRITON_ATTN",
    dtype="float16" 
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

2026-04-21 19:24:15.618563: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776799455.839187      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776799455.902533      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776799456.426286      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776799456.426333      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776799456.426336      55 computation_placer.cc:177] computation placer alr

INFO 04-21 19:24:40 [utils.py:261] non-default args: {'dtype': 'float16', 'max_model_len': 16384, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'attention_backend': 'TRITON_ATTN', 'model': 'Qwen/Qwen3-4B-Instruct-2507'}


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

INFO 04-21 19:25:02 [model.py:541] Resolved architecture: Qwen3ForCausalLM
WARNING 04-21 19:25:02 [model.py:1885] Casting torch.bfloat16 to torch.float16.
INFO 04-21 19:25:02 [model.py:1561] Using max model len 16384
INFO 04-21 19:25:03 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-21 19:25:03 [vllm.py:624] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

...

(Worker_TP0 pid=283) INFO 04-21 19:26:59 [default_loader.py:291] Loading weights took 5.27 seconds
(Worker_TP0 pid=283) INFO 04-21 19:27:00 [gpu_model_runner.py:4118] Model loading took 3.87 GiB memory and 77.510145 seconds
(Worker_TP0 pid=283) INFO 04-21 19:27:15 [backends.py:805] Using cache directory: /root/.cache/vllm/torch_compile_cache/940cfad38a/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=283) INFO 04-21 19:27:15 [backends.py:865] Dynamo bytecode transform time: 15.05 s


(Worker_TP0 pid=283) [rank0]:W0421 19:27:28.664000 283 torch/_inductor/utils.py:1613] Not enough SMs to use max_autotune_gemm mode
(Worker_TP1 pid=284) [rank1]:W0421 19:27:31.683000 284 torch/_inductor/utils.py:1613] Not enough SMs to use max_autotune_gemm mode


(Worker_TP0 pid=283) INFO 04-21 19:27:39 [backends.py:302] Cache the graph of compile range (1, 8192) for later use
(Worker_TP1 pid=284) INFO 04-21 19:27:40 [backends.py:302] Cache the graph of compile range (1, 8192) for later use
(Worker_TP0 pid=283) INFO 04-21 19:27:52 [backends.py:319] Compiling a graph for compile range (1, 8192) takes 25.96 s
(Worker_TP0 pid=283) INFO 04-21 19:27:52 [monitor.py:34] torch.compile takes 41.01 s in total
(Worker_TP0 pid=283) INFO 04-21 19:27:55 [gpu_worker.py:356] Available KV cache memory: 7.05 GiB
(EngineCore_DP0 pid=255) INFO 04-21 19:27:55 [kv_cache_utils.py:1307] GPU KV cache size: 102,720 tokens
(EngineCore_DP0 pid=255) INFO 04-21 19:27:55 [kv_cache_utils.py:1312] Maximum concurrency for 16,384 tokens per request: 6.27x


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:08<00:00,  6.10it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:15<00:00,  2.19it/s]


(Worker_TP0 pid=283) INFO 04-21 19:28:21 [gpu_model_runner.py:5051] Graph capturing finished in 26 secs, took 1.28 GiB
(EngineCore_DP0 pid=255) INFO 04-21 19:28:21 [core.py:272] init engine (profile, create kv cache, warmup model) took 81.48 seconds
(EngineCore_DP0 pid=255) INFO 04-21 19:28:26 [vllm.py:624] Asynchronous scheduling is enabled.
INFO 04-21 19:28:26 [llm.py:343] Supported tasks: ['generate']


In [6]:
secrets = UserSecretsClient()
db_params = {
    "dbname": "arxivdb",
    "user": secrets.get_secret("DB_USER"),
    "password": secrets.get_secret("DB_PASSWORD"),
    "host": secrets.get_secret("DB_HOST"),
    "port": "5433"  
}


In [7]:
!pip install lancedb sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 MB 33.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 21.4 MB/s eta 0:00:00


In [8]:
import lancedb
from sentence_transformers import SentenceTransformer

db_path = "/kaggle/input/datasets/fluloeo/atable/arxiv_lancedb"

if os.path.exists(db_path):
    db = lancedb.connect(db_path)
    table = db.open_table("papers")
    print(f"База успешно подключена! Количество записей: {len(table)}")
else:
    print("Ошибка: Путь не найден. Проверьте название датасета в правой панели.")


retrieval_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')


База успешно подключена! Количество записей: 720028


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
import warnings
import os

# 1. Отключаем предупреждения на уровне интерпретатора
os.environ['PYTHONWARNINGS'] = 'ignore'

# 2. Фильтруем конкретное сообщение о utcnow (оно самое назойливое в 3.12)
warnings.filterwarnings("ignore", message=".*datetime\.datetime\.utcnow.*", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*datetime\.datetime\.utcfromtimestamp.*", category=DeprecationWarning)

# 3. Общий игнор (на всякий случай)
warnings.filterwarnings("ignore", category=DeprecationWarning)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [16]:
import datetime
from datetime import timezone

# Подменяем старый utcnow на новый формат, который не выдает warning
def fixed_utcnow():
    return datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)

datetime.datetime.utcnow = fixed_utcnow

TypeError: cannot set 'utcnow' attribute of immutable type 'datetime.datetime'

In [18]:

from processing import ArticleProcessor
from summarization import SummarizationPipeline, VLLMProvider
from agent import ArxivAgent, LanceDBRetriever
from vllm import SamplingParams



hub_prompts = {
    "classifier": "fluloeo/arxiv-classifier",
    "rewriter": "fluloeo/arxiv-rewriter",
    "qa": "fluloeo/arxiv-qa",
    "summarization": {
        "map": "fluloeo/arxiv-summarizer-map",
        "reduce": "fluloeo/arxiv-summarizer-reduce"
    }
}

USE_HUB = True # Переключите на False для использования только локальных yaml-промптов
provider = VLLMProvider(llm, SamplingParams, "Qwen/Qwen3-4B-Instruct-2507")
processor = ArticleProcessor(tokenizer)
sum_pipe = SummarizationPipeline(provider, tokenizer, hub_prompts['summarization'], None, USE_HUB)
retriever = LanceDBRetriever(table)


agent = ArxivAgent(
    llm_provider=provider,
    retriever=retriever,
    sum_pipeline=sum_pipe,
    processor=processor,
    embed_model=retrieval_model,
    db_params=db_params,
    tokenizer=tokenizer,
    prompts=hub_prompts, 
    local_prompts=None,
    use_hub=USE_HUB
)






Pulling prompt from LangSmith: fluloeo/arxiv-summarizer-map
Pulling prompt from LangSmith: fluloeo/arxiv-summarizer-reduce
Pulling prompt from LangSmith: fluloeo/arxiv-classifier
Pulling prompt from LangSmith: fluloeo/arxiv-rewriter
Pulling prompt from LangSmith: fluloeo/arxiv-qa


In [23]:
query = "Какие алгоритмы используются для моделирования жидкости?"
result = agent.invoke(query)
display(Markdown(result['final_answer']))

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/9 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

# a survey of ocean simulation and rendering techniques in computer graphics
🔗 [PDF](https://export.arxiv.org/pdf/1109.6494.pdf)

**Аналитический отчёт по моделированию динамики и визуализации океанической поверхности**

---

**Цель исследования**  
Разработка реалистичных и физически обоснованных моделей динамики океана, включающих как геометрическое поведение поверхности (волны, разрушение в мелких водах), так и оптические свойства (цвет, пена, брызги, взаимодействие света с водой). Основная задача — синтезировать интегрированный подход, сочетающий физические модели, адаптивную визуализацию и эффективные вычислительные схемы для интерактивного представления океанских явлений в реальном времени.

---

**Методология**  
Работа строится на комбинации нескольких подходов:  
- **Параметрические и спектральные методы** используют сумму синусоидальных функций для моделирования поверхности в глубокой воде. Параметры волн (амплитуда $ A_i $, волновой вектор $ \mathbf{k}_i $, частота $ \omega_i $) определяются на основе теории Айри и спектров волн (Pierson-Moskowitz, JONSWAP). Амплитуда компоненты в JONSWAP задаётся как:  
  $$
  h(k,0) = \sqrt{\frac{2}{\pi} \cdot \frac{g}{\alpha} \cdot \frac{1}{k} \cdot \exp\left(-\frac{k^2}{k_0^2}\right) \cdot \left(1 + \gamma \cdot \exp\left(-\frac{(k - k_m)^2}{2\sigma^2}\right)\right)}
  $$  
  где $ \alpha = 0.0081 $ — постоянная Phillips, $ k_m $ — частота пика, $ f_m = \sqrt{\frac{g}{2\pi L}} $, $ L = U_{10}^2 / g $.  
- **Физически обоснованные модели** (на основе уравнений Навье–Стокса) применяются вблизи берега, где взаимодействие с дном критично. Эйлеровский метод решает уравнения в воксельной сетке, а лагранжевый подход (SPH, частицы) моделирует брызги, пену и пузыри.  
- **Гибридные схемы** комбинируют эйлеровское моделирование основного течения с лагранжевым генерированием мелких деталей в зонах высокой кривизны.  
- **Оптимизация на GPU** включает адаптивную тесселяцию, использование Perlin-шума для смещения, отбрасывание высоких частот в удалённых областях и применение концепта *grid project* для снижения разрешения.  

---

**Результаты**  
- Реализация спектральных моделей обеспечивает высокую точность воспроизведения волнового спектра в зависимости от скорости ветра.  
- Методы на основе NSE позволяют точно моделировать разрушение волн в мелких водах, в то время как параметрические методы остаются эффективными в глубокой воде.  
- Модели пенки и брызг, основанные на эмпирических зависимостях (например, от скорости ветра >13 км/ч) и частицных системах, обеспечивают визуальную реалистичность, хотя требуют улучшения синхронизации с волнами.  
- Взаимодействие света с водой моделируется с помощью первого порядка (геометрическая оптика, Beer-Lambert закон: $ I(d) = I(0) \cdot e^{-a d} $) и многоуровневого приближения (уравнение радиационного переноса), что позволяет учитывать поглощение, рассеяние и отражение.  

---

**Заключение**  
Предложенная архитектура объединяет физическую достоверность, визуальную реалистичность и вычислительную эффективность. Она позволяет создавать интерактивные симуляции океана, адаптированные к масштабу отображения, с учётом глубины, ветра, оптических свойств и динамики поверхностных явлений. Работа ложится в основу современных систем визуализации природных сред, особенно в играх, виртуальной реальности и научных визуализациях.

In [24]:
processor.visualize(result['debug_data'])

**Всего фрагментов:** `9` | **Длина:** `13016` симв.

---

### *Chunk 1*: introduction + ocean dynamics simulation in deep water
>`Символов: 1471`


---


### *Chunk 2*: early works
>`Символов: 1373`


---


...


---


### *Chunk 8*: realistic ocean surface rendering and lighting + foam and spray + empirical models + model + particle systems
>`Символов: 1495`


---


### *Chunk 9*: light water interactions + first order approximation + multiple order approximation + conclusion
>`Символов: 1506`


---


In [12]:
result = agent.invoke("что такое dropout?")
display(Markdown(result['final_answer']))

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

**Dropout** — это техника регуляризации в нейронных сетях, при которой во время обучения случайным образом отключаются («выключаются») некоторые нейроны (или их соединения) с заданной вероятностью. Это помогает предотвратить **переобучение** (overfitting), поскольку сеть вынуждена учиться на более обобщённых признаках, не завися от конкретных нейронов.

В контексте текста:

- Dropout **работает лучше, чем методы, основанные на байесовском среднем**, особенно в сетях с одним скрытым слоем.
- При этом **вероятность отключения (dropout rate)** обычно устанавливается на уровне **0.5** (то есть 50% нейронов отключаются), поскольку это показывает лучшую производительность и устойчивость.
- Важно, что **входные данные** также могут быть обработаны с dropout, хотя часто лучше сохранять более 50% входов.
- Можно адаптировать вероятность dropout для отдельных нейронов, сравнивая их влияние на производительность на валидационном наборе.
- В сложных задачах (например, при наличии разных режимов отображения данных) можно сделать dropout-вероятность **функцией от входа**, что создаёт эффективный «мешок экспертов» (mixture of experts), где каждый нейрон работает как эксперт в определённой области.
- В отличие от **байесовского среднего**, которое требует сложных методов (например, Монте-Карло симуляций) для оценки постериорного распределения моделей, dropout **прост в реализации** и эффективен: при тестировании достаточно одного прохода по сети с включённым dropout, чтобы получить среднее предсказание, что делает его **высокопроизводительным и быстрым**.
- Dropout **улучшает обучение признаков**: нейроны учатся более простым, интерпретируемым признакам (например, линиям, контурам), а не зависят от других нейронов (что уменьшает **ко-адаптацию**).
- В экспериментах на MNIST, TIMIT, CIFAR-10 и ImageNet dropout **существенно снижает количество ошибок** по сравнению с обычной обратной связью (backpropagation), улучшает обобщение и делает обучение более стабильным.

Таким образом, **dropout — это простая, эффективная и мощная техника регуляризации, которая заставляет сеть учиться более устойчивым и обобщённым способом, уменьшая риск переобучения и улучшая производительность на новых данных**.

In [21]:
from langchainhub.client import Client
client = Client()
push = client.push

In [ ]:
# import os
# import json
# from kaggle_secrets import UserSecretsClient
# from langchain_core.load import dumpd
# from langchain_core.prompts import ChatPromptTemplate
# from langchainhub.client import Client

# # 1. Настройка доступа
# secrets = UserSecretsClient()
# os.environ["LANGCHAIN_API_KEY"] = secrets.get_secret("LANGSMITH_API_KEY")

# hub_client = Client()
# # ВАЖНО: Замените 'fluloeo' на ваш Handle в LangSmith, если он отличается
# USER_HANDLE = "fluloeo@gmail.com"

# def push_to_hub(repo_name, prompt_obj):
#     """
#     Отправляет промпт в Хаб. 
#     Если передать просто 'arxiv-qa', библиотека сама превратит это 
#     в 'ваш_handle/arxiv-qa'.
#     """
#     try:
#         # Сериализуем объект в JSON
#         manifest_json = json.dumps(dumpd(prompt_obj))
        
#         # Передаем ТОЛЬКО название репозитория (без слэша и хендла)
#         hub_client.push(repo_name, manifest_json)
#         print(f"✅ Успешно загружено: {repo_name}")
#     except Exception as e:
#         print(f"❌ Ошибка при загрузке {repo_name}: {e}")

# # --- ОПРЕДЕЛЕНИЕ ПРОМПТОВ ---

# # 1. Summarization: MAP
# map_prompt = ChatPromptTemplate.from_messages([
#     ("system", """Ты — профессиональный научный редактор. Твоя задача — извлечь ключевую техническую информацию из фрагмента научной статьи.
# ПРАВИЛА:
# 1. Используй только предоставленный текст.
# 2. Сохраняй строгую техническую терминологию.
# 3. Пиши на РУССКОМ языке, если есть плохо переводимые термины, сохраняй их на оригинальном языке.
# 4. Текст в блоках [КОНТЕКСТ] дан только для понимания связей — НЕ ВКЛЮЧАЙ информацию из них в суммаризацию.
# 5. Формат: плотный список тезисов (bullet points).
# 6. Никаких вступлений — только факты.
# 7. Если есть математические формулы или символы, выводи их через LaTeX (используя символы $, которые будут понятны движку MathJax)"""),
#     ("user", """Ниже представлен раздел статьи: {title}
# [КОНТЕКСТ ПРОШЛОГО]: {past_overlap}
# [ОСНОВНОЙ ТЕКСТ]: {main_text}
# [КОНТЕКСТ БУДУЩЕГО]: {future_overlap}

# Извлеки цели, методы и ключевые данные из [ОСНОВНОГО ТЕКСТА].""")
# ])

# # 2. Summarization: REDUCE
# reduce_prompt = ChatPromptTemplate.from_messages([
#     ("system", """Ты — ведущий научный эксперт. Твоя задача — синтезировать единый, связный аналитический обзор на основе предоставленных кратких выжимок из разных разделов статьи.
# ТРЕБОВАНИЯ:
# 1. Язык: РУССКИЙ (академический стиль). Сохраняй термины на оригинальном языке, если нет однозначного перевода.
# 2. Структура строго по разделам:
# - **Цель исследования**: проблема и задачи.
# - **Методология**: технический стек, алгоритмы, эксперименты.
# - **Результаты**: конкретные показатели, улучшения, находки.
# - **Заключение**: значимость работы.
# 3. БЕЗ длинных формул, которые не отражают основную суть.
# 4. Математические символы, без которых невозможно достоверно донести суть, оформляй как LaTeX формулу для markdown.
# 5. Объем: 400-600 слов. Высокая плотность информации.
# 6. Не добавляй ничего от себя, чего нет в исходных данных."""),
#     ("user", "Вот краткие выжимки по разделам статьи:\n\n{summaries}\n\nСинтезируй на их основе финальный аналитический отчет.")
# ])

# # 3. Classifier
# classifier_prompt = ChatPromptTemplate.from_messages([
#     ("system", "Ты — классификатор запросов к базе научных статей ArXiv. Отвечай строго по формату."),
#     ("user", """Классифицируй запрос пользователя.
# Если пользователь хочет обзор, суммаризацию или поиск нескольких статей по теме — ответь 'YES'.
# Если пользователь задает конкретный вопрос, требующий точного ответа по фактам из одной статьи — ответь 'NO'.

# Запрос: {query}
# Ответ (начиная с YES или NO):""")
# ])

# # 4. Rewriter
# rewriter_prompt = ChatPromptTemplate.from_messages([
#     ("system", "Ты — эксперт по научному поиску в базе данных ArXiv. Твоя задача: переформулировать запрос пользователя на английский язык для векторного поиска."),
#     ("user", """ПРАВИЛА:
# 1. Используй только научную терминологию.
# 2. Переводи на английский язык.
# 3. Удали команды 'суммаризируй', 'найди', 'сделай обзор'.
# 4. Выдай ТОЛЬКО текст запроса.

# Запрос для оптимизации: {query}""")
# ])

# # 5. QA
# qa_prompt = ChatPromptTemplate.from_messages([
#     ("system", "Ты — эксперт-научный ассистент. Отвечай на русском языке, максимально точно используя предоставленный контекст."),
#     ("user", """Контекст:
# {context}

# Вопрос: {query}
# Ответ:""")
# ])

# # --- ЗАГРУЗКА ---

# print("Начинаю загрузку промптов в LangSmith Hub...")
# push_to_hub("arxiv-summarizer-map", map_prompt)
# push_to_hub("arxiv-summarizer-reduce", reduce_prompt)
# push_to_hub("arxiv-classifier", classifier_prompt)
# push_to_hub("arxiv-rewriter", rewriter_prompt)
# push_to_hub("arxiv-qa", qa_prompt)
# print("\n🚀 Все промпты в облаке!")

In [ ]:
import os

REPO_PATH = "/kaggle/working/Summarization-scientific-articles"

if os.path.exists(REPO_PATH):
    print("Обновляю репозиторий...")
    # --git-dir и --work-tree гарантируют, что git поймет, где лежат файлы
    !git -C {REPO_PATH} fetch --all
    !git -C {REPO_PATH} reset --hard origin/main  
else:
    print("Репозиторий не найден. Клонирую заново...")
    !git clone https://github.com/fluloeo/Summarization-scientific-articles.git {REPO_PATH}

import sys
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

import importlib
import agent, processing, summarization

importlib.reload(processing)
importlib.reload(summarization)
importlib.reload(agent)

from agent import ArxivAgent, LanceDBRetriever
print("✅ Код успешно обновлен!")